**Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset

!pip install evaluate
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

import torch

Reading Dataset

In [ ]:
df = pd.read_csv("MESC.csv")

# Filter missing values
df = df.dropna(subset=["Utterance", "Emotion"])
df.head()


,Utterance,Speaker,Emotion,Strategy,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,I told you.,Client,sadness,undefined,0,0,1,1,"00:00:54,097","00:00:55,113"
1,Told me what?,Therapist,neutral,Open question,0,1,1,1,"00:00:55,260","00:00:56,263"
2,That you'd be sorry you ever encouraged me to ...,Client,sadness,undefined,0,2,1,1,"00:00:56,651","00:00:59,719"
3,I'm not sorry at all.,Therapist,neutral,Communication Skills,0,3,1,1,"00:00:59,951","00:01:01,237"
4,"You didn't expect it to be like this, I bet.",Client,sadness,undefined,0,4,1,1,"00:01:03,404","00:01:05,206"


Oversampling

In [ ]:
from collections import Counter
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 42

X_train = train_df["Utterance"]
y_train = train_df["label"]


In [ ]:
print("\n" + "="*60)
print("OVERSAMPLING MINORITY CLASSES")
print("="*60)

print("\nBefore oversampling:")
train_class_counts = Counter(y_train)
for emotion, count in train_class_counts.items():
    print(f"  {le.inverse_transform([emotion])[0]}: {count}")

# Apply random oversampling
ros = RandomOverSampler(random_state=RANDOM_STATE)
# Resample using indices to maintain connection to original X_train
X_train_indices = np.arange(len(X_train)).reshape(-1, 1)
X_train_indices_resampled, y_train_resampled = ros.fit_resample(X_train_indices, y_train)

X_train_oversampled = X_train.iloc[X_train_indices_resampled.flatten()]
y_train_oversampled = y_train_resampled

print("\nAfter oversampling:")
train_class_counts_oversampled = Counter(y_train_oversampled)
for emotion, count in train_class_counts_oversampled.items():
    print(f"  {le.inverse_transform([emotion])[0]}: {count}")

print(f"\nTraining set size increased: {len(X_train)} \u2192 {len(X_train_oversampled)}")


OVERSAMPLING MINORITY CLASSES

Before oversampling:
  depression: 4462
  joy: 1215
  neutral: 15404
  anger: 2393
  sadness: 683
  disgust: 1502
  fear: 226

After oversampling:
  depression: 15404
  joy: 15404
  neutral: 15404
  anger: 15404
  sadness: 15404
  disgust: 15404
  fear: 15404

Training set size increased: 25885 → 107828


In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Emotion"])
num_labels = len(le.classes_)
print("Classes:", le.classes_)

Classes: ['anger' 'depression' 'disgust' 'fear' 'joy' 'neutral' 'sadness']


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["Utterance"],
        truncation=True,
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/25885 [00:00<?, ? examples/s]

Map:   0%|          | 0/2877 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [ ]:
import transformers
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print("TrainingArguments module:", TrainingArguments.__module__)

Transformers version: 4.57.2
TrainingArguments module: transformers.training_args


In [ ]:
training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    save_steps=99999999,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-3240721288.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.160100,1.129972,0.602364
2,1.078500,1.148891,0.600973
3,0.728100,1.293769,0.581856
4,0.520200,1.491927,0.570733


TrainOutput(global_step=6472, training_loss=0.8951819379191168, metrics={'train_runtime': 561.7802, 'train_samples_per_second': 184.307, 'train_steps_per_second': 11.521, 'total_flos': 1000062181863000.0, 'train_loss': 0.8951819379191168, 'epoch': 4.0})

Result

In [ ]:
results = trainer.evaluate()
results

{'eval_loss': 1.4919267892837524,
 'eval_accuracy': 0.5707334028501911,
 'eval_runtime': 3.5728,
 'eval_samples_per_second': 805.252,
 'eval_steps_per_second': 50.381,
 'epoch': 4.0}